<a href="https://colab.research.google.com/github/thomassd17/INF220-EstructuraDeDatos1/blob/thomassd17-patch-1/DesafiosUnidad1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unidad 1: Tipos de Datos Abstractos (TDA) y Gestión de Memoria
## 6 Desafíos — Resolución de Problemas

**Estudiante:** thomassd17

Cada desafío sigue el mismo formato:
1. Código **con el error** (tal como fue entregado).
2. **Explicación** del error conceptual.
3. Código **corregido**.
4. **Prueba** que demuestra el comportamiento correcto.


---
## Desafío 1: El misterio de las referencias (Gestión de Memoria)

**Problema:** Un estudiante intenta duplicar una lista de valores, pero al modificar la "copia", la original también cambia. ¿Por qué sucede esto y cómo se arregla?

In [ ]:
# CODIGO CON ERROR
original_data = [10, 20, 30]

# El estudiante intenta copiar los datos
copy_data = original_data

copy_data.append(40)

print(f"Original: {original_data}")
print(f"Copia: {copy_data}")

# Ambas listas muestran el 40, aunque solo se modifico 'copy_data'


Original: [10, 20, 30, 40]
Copia: [10, 20, 30, 40]


**¿Por qué sucede?** En Python, las listas son objetos **mutables**. La línea `copy_data = original_data` **no crea una lista nueva**: solo crea una segunda etiqueta (variable) que apunta al **mismo objeto en el Heap**. Por eso, al modificar `copy_data`, en realidad se está modificando el único objeto que existe, y `original_data` "ve" el cambio porque apunta a esa misma dirección de memoria.

**Corrección:** hay que crear una copia real e independiente, usando `list(...)`, el método `.copy()`, o el slicing `[:]`.

In [ ]:
# CODIGO CORREGIDO
original_data = [10, 20, 30]

# Ahora si se crea una copia independiente (nuevo objeto en el Heap)
copy_data = original_data.copy()   # tambien valido: list(original_data) o original_data[:]

copy_data.append(40)

print(f"Original: {original_data}")
print(f"Copia: {copy_data}")

assert original_data == [10, 20, 30], "La original no deberia cambiar"
assert copy_data == [10, 20, 30, 40]
print("Correcto: la copia es independiente de la original")


Original: [10, 20, 30]
Copia: [10, 20, 30, 40]
Correcto: la copia es independiente de la original


---
## Desafío 2: Implementación de TDA Pila (Stack)

**Problema:** La clase `Stack` tiene errores críticos: `is_empty` tiene la lógica invertida, `push` no guarda el elemento recibido (lo reemplaza), y `pop` ni valida que haya elementos ni retorna el elemento eliminado.

In [ ]:
# CODIGO CON ERROR
class StackConError:
    def __init__(self):
        self.items = []

    def is_empty(self):
        # ERROR: Logica invertida
        return len(self.items) > 0

    def push(self, item):
        # ERROR: No esta agregando el item recibido
        self.items = [item]

    def pop(self):
        # ERROR: No verifica si hay elementos antes de borrar
        # y no retorna el elemento eliminado
        self.items.remove(-1)

# --- Prueba del Alumno ---
mi_pila = StackConError()
mi_pila.push("A")
mi_pila.push("B")

print("Items reales tras dos push:", mi_pila.items)  # solo queda ['B'], se perdio 'A'
print("Esta vacia? (deberia ser False):", mi_pila.is_empty())  # da True: logica invertida


Items reales tras dos push: ['B']
Esta vacia? (deberia ser False): True


**¿Por qué falla cada parte?**
- `is_empty`: devuelve `True` cuando **hay** elementos y `False` cuando **no hay** — está exactamente al revés.
- `push`: `self.items = [item]` **reemplaza toda la lista** por una lista de un solo elemento, en vez de usar `.append(item)` para agregar sin perder lo anterior.
- `pop`: `self.items.remove(-1)` intenta **eliminar el valor `-1`** de la lista (no el último elemento), no valida si la pila está vacía, y no retorna nada — un `pop` de una pila (LIFO) debe quitar y devolver el **último** elemento agregado.

**Corrección:**

In [ ]:
# CODIGO CORREGIDO
class Stack:
    def __init__(self):
        self.items = []

    def is_empty(self):
        return len(self.items) == 0

    def push(self, item):
        self.items.append(item)

    def pop(self):
        if self.is_empty():
            raise IndexError("No se puede hacer pop: la pila esta vacia")
        return self.items.pop()  # quita y retorna el ultimo elemento (LIFO)

# --- Prueba corregida ---
mi_pila = Stack()
mi_pila.push("A")
mi_pila.push("B")

print("Esta vacia?", mi_pila.is_empty())
print("Elemento sacado:", mi_pila.pop())
print("Elemento sacado:", mi_pila.pop())

try:
    mi_pila.pop()
except IndexError as e:
    print("Correcto, controla la pila vacia:", e)


Esta vacia? False
Elemento sacado: B
Elemento sacado: A
Correcto, controla la pila vacia: No se puede hacer pop: la pila esta vacia


---
## Desafío 3: Gestión de Memoria en el Heap (Nodos Dinámicos)

**Problema:** El código intenta crear una secuencia `[Nodo 1] -> [Nodo 2]` en el Heap, pero al reasignar la variable `contenedor` directamente a un nuevo `Nodo`, se pierde la referencia al primer nodo en lugar de enlazarlos.

In [ ]:
# CODIGO CON ERROR
class Nodo:
    def __init__(self, valor):
        self.valor = valor
        self.siguiente = None  # 'puntero' al siguiente espacio en el Heap

# --- Prueba del Alumno ---
contenedor = Nodo("Datos Importantes 1")

# ERROR: el alumno intenta agregar el segundo nodo de esta forma
contenedor = Nodo("Datos Importantes 2")

# Verificacion
print(f"Contenido actual: {contenedor.valor}")
if contenedor.siguiente is None:
    print("ERROR: Se ha perdido la referencia al primer nodo. Memory leak conceptual!")


Contenido actual: Datos Importantes 2
ERROR: Se ha perdido la referencia al primer nodo. Memory leak conceptual!


**¿Por qué falla?** `contenedor = Nodo("Datos Importantes 2")` **reasigna la variable `contenedor`** para que apunte a un objeto completamente nuevo en el Heap. La referencia al primer `Nodo` (`"Datos Importantes 1"`) se pierde por completo: nadie apunta ya hacia él, así que es inaccesible (equivalente conceptual a una fuga de memoria).

**Corrección:** en vez de reasignar `contenedor`, hay que usar el campo `siguiente` del primer nodo para **enlazar** el segundo, manteniendo `contenedor` apuntando siempre al inicio de la cadena.

In [ ]:
# CODIGO CORREGIDO
class Nodo:
    def __init__(self, valor):
        self.valor = valor
        self.siguiente = None

# --- Prueba corregida ---
contenedor = Nodo("Datos Importantes 1")

# Se crea el segundo nodo y se ENLAZA desde el primero, sin perder la referencia inicial
contenedor.siguiente = Nodo("Datos Importantes 2")

print(f"Nodo inicial: {contenedor.valor}")
print(f"Nodo siguiente: {contenedor.siguiente.valor}")

assert contenedor.valor == "Datos Importantes 1"
assert contenedor.siguiente.valor == "Datos Importantes 2"
print("Correcto: 'contenedor' sigue apuntando al inicio y la cadena esta completa")


Nodo inicial: Datos Importantes 1
Nodo siguiente: Datos Importantes 2
Correcto: 'contenedor' sigue apuntando al inicio y la cadena esta completa


---
## Desafío 4: TDA Punto y Circunferencia (Paso por Referencia vs. Valor)

**Problema:** Al modificar directamente el `Punto` original usado como centro de una `Circunferencia`, la circunferencia también cambia, porque solo guarda una referencia al mismo objeto en vez de una copia de sus valores.

In [ ]:
# CODIGO CON ERROR
class Punto:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"({self.x}, {self.y})"

class CircunferenciaConError:
    def __init__(self, centro, radio):
        self.centro = centro  # ERROR: se asigna la referencia directa al objeto Punto
        self.radio = radio

    def __str__(self):
        return f"Circunferencia con centro en {self.centro} y radio {self.radio}"

# --- Prueba del Alumno ---
mi_punto_original = Punto(1, 2)
mi_circunferencia = CircunferenciaConError(mi_punto_original, 5)

print(f"Circunferencia inicial: {mi_circunferencia}")

mi_punto_original.x = 10
mi_punto_original.y = 20

print(f"Circunferencia despues de modificar el punto original: {mi_circunferencia}")


Circunferencia inicial: Circunferencia con centro en (1, 2) y radio 5
Circunferencia despues de modificar el punto original: Circunferencia con centro en (10, 20) y radio 5


**¿Por qué falla?** `self.centro = centro` guarda la **misma referencia** al objeto `Punto` que se pasó como argumento; no crea un `Punto` nuevo. Como `Punto` es mutable, cualquier cambio sobre `mi_punto_original` se refleja automáticamente en `mi_circunferencia.centro`, porque en realidad son el mismo objeto en memoria.

**Corrección:** la `Circunferencia` debe crear **su propia copia** de las coordenadas del centro (por ejemplo, un `Punto` nuevo con los mismos valores `x`, `y`), en vez de guardar la referencia recibida.

In [ ]:
# CODIGO CORREGIDO
class Punto:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"({self.x}, {self.y})"

class Circunferencia:
    def __init__(self, centro, radio):
        # Se crea una copia independiente del centro (nuevo objeto Punto)
        self.centro = Punto(centro.x, centro.y)
        self.radio = radio

    def __str__(self):
        return f"Circunferencia con centro en {self.centro} y radio {self.radio}"

# --- Prueba corregida ---
mi_punto_original = Punto(1, 2)
mi_circunferencia = Circunferencia(mi_punto_original, 5)

print(f"Circunferencia inicial: {mi_circunferencia}")

mi_punto_original.x = 10
mi_punto_original.y = 20

print(f"Circunferencia despues de modificar el punto original: {mi_circunferencia}")

assert mi_circunferencia.centro.x == 1 and mi_circunferencia.centro.y == 2
print("Correcto: la circunferencia mantiene su propio centro independiente")


Circunferencia inicial: Circunferencia con centro en (1, 2) y radio 5
Circunferencia despues de modificar el punto original: Circunferencia con centro en (1, 2) y radio 5
Correcto: la circunferencia mantiene su propio centro independiente


---
## Desafío 5: TDA Automóvil y Garaje (Gestión de Objetos en Colecciones)

**Problema:** Al reutilizar la misma variable de objeto `Automovil` para "crear" un segundo auto (modificando sus atributos en vez de instanciar uno nuevo), el garaje termina con el mismo objeto repetido en la lista.

In [ ]:
# CODIGO CON ERROR
class Automovil:
    def __init__(self, marca, modelo, color):
        self.marca = marca
        self.modelo = modelo
        self.color = color

    def __str__(self):
        return f"{self.color} {self.marca} {self.modelo}"

class Garaje:
    def __init__(self):
        self.automoviles = []

    def agregar_automovil(self, auto):
        self.automoviles.append(auto)

    def mostrar_automoviles(self):
        print("Automoviles en el garaje:")
        for i, auto in enumerate(self.automoviles):
            print(f"  {i+1}. {auto}")

# --- Prueba del Alumno ---
mi_garaje = Garaje()

auto_temporal = Automovil("Toyota", "Corolla", "Rojo")
mi_garaje.agregar_automovil(auto_temporal)

# ERROR: reutiliza la misma variable/objeto en vez de crear uno nuevo
auto_temporal.marca = "Honda"
auto_temporal.modelo = "Civic"
auto_temporal.color = "Azul"
mi_garaje.agregar_automovil(auto_temporal)

mi_garaje.mostrar_automoviles()
# Muestra "Azul Honda Civic" DOS veces: la lista guarda la MISMA referencia en ambas posiciones


Automoviles en el garaje:
  1. Azul Honda Civic
  2. Azul Honda Civic


**¿Por qué falla?** `mi_garaje.automoviles.append(auto_temporal)` guarda la **referencia** al objeto `auto_temporal`, no una copia de sus valores actuales. Como después se modifican los atributos de ese mismo objeto (`auto_temporal.marca = "Honda"`, etc.) en vez de crear un `Automovil` nuevo, **ambas posiciones de la lista apuntan al mismo objeto**, y por eso las dos entradas muestran los datos del Honda Civic azul.

**Corrección:** crear una instancia nueva de `Automovil` para cada auto que se quiera agregar, en vez de modificar y reutilizar la misma variable.

In [ ]:
# CODIGO CORREGIDO
class Automovil:
    def __init__(self, marca, modelo, color):
        self.marca = marca
        self.modelo = modelo
        self.color = color

    def __str__(self):
        return f"{self.color} {self.marca} {self.modelo}"

class Garaje:
    def __init__(self):
        self.automoviles = []

    def agregar_automovil(self, auto):
        self.automoviles.append(auto)

    def mostrar_automoviles(self):
        print("Automoviles en el garaje:")
        for i, auto in enumerate(self.automoviles):
            print(f"  {i+1}. {auto}")

# --- Prueba corregida ---
mi_garaje = Garaje()

auto_1 = Automovil("Toyota", "Corolla", "Rojo")
mi_garaje.agregar_automovil(auto_1)

auto_2 = Automovil("Honda", "Civic", "Azul")  # instancia NUEVA, no reutiliza auto_1
mi_garaje.agregar_automovil(auto_2)

mi_garaje.mostrar_automoviles()

assert str(mi_garaje.automoviles[0]) == "Rojo Toyota Corolla"
assert str(mi_garaje.automoviles[1]) == "Azul Honda Civic"
print("Correcto: el garaje contiene dos automoviles distintos e independientes")


Automoviles en el garaje:
  1. Rojo Toyota Corolla
  2. Azul Honda Civic
Correcto: el garaje contiene dos automoviles distintos e independientes


---
## Desafío 6: TDA Línea (Inmutabilidad Conceptual con Objetos Mutables)

**Problema:** La clase `Linea` guarda referencias directas a los objetos `Punto` recibidos. Si se modifica un `Punto` original después de crear la línea, la línea cambia con él, cuando conceptualmente debería quedar fija una vez creada.

In [ ]:
# CODIGO CON ERROR
class Punto:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"({self.x}, {self.y})"

class LineaConError:
    def __init__(self, inicio, fin):
        self.inicio = inicio  # ERROR: almacena la referencia directa
        self.fin = fin        # ERROR: almacena la referencia directa

    def __str__(self):
        return f"Linea de {self.inicio} a {self.fin}"

# --- Prueba del Alumno ---
punto_a = Punto(0, 0)
punto_b = Punto(5, 5)

mi_linea = LineaConError(punto_a, punto_b)
print(f"Linea original: {mi_linea}")

punto_a.x = 10
punto_a.y = 10

print(f"Linea despues de modificar el punto 'A': {mi_linea}")


Linea original: Linea de (0, 0) a (5, 5)
Linea despues de modificar el punto 'A': Linea de (10, 10) a (5, 5)


**¿Por qué falla?** Es el mismo problema conceptual del Desafío 4: `self.inicio = inicio` y `self.fin = fin` guardan la **referencia** a los objetos `Punto` originales, no una copia. Como `Punto` es mutable, modificar `punto_a` después de crear `mi_linea` afecta directamente a `mi_linea.inicio`, porque son el mismo objeto en el Heap.

**Corrección:** la `Linea` debe crear sus propias copias de `inicio` y `fin` en el momento de construirse, para quedar desacoplada de los objetos `Punto` originales.

In [ ]:
# CODIGO CORREGIDO
class Punto:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"({self.x}, {self.y})"

class Linea:
    def __init__(self, inicio, fin):
        # Se crean copias independientes de los puntos recibidos
        self.inicio = Punto(inicio.x, inicio.y)
        self.fin = Punto(fin.x, fin.y)

    def __str__(self):
        return f"Linea de {self.inicio} a {self.fin}"

# --- Prueba corregida ---
punto_a = Punto(0, 0)
punto_b = Punto(5, 5)

mi_linea = Linea(punto_a, punto_b)
print(f"Linea original: {mi_linea}")

punto_a.x = 10
punto_a.y = 10

print(f"Linea despues de modificar el punto 'A': {mi_linea}")

assert mi_linea.inicio.x == 0 and mi_linea.inicio.y == 0
print("Correcto: la linea conserva sus puntos originales, sin importar cambios externos")


Linea original: Linea de (0, 0) a (5, 5)
Linea despues de modificar el punto 'A': Linea de (0, 0) a (5, 5)
Correcto: la linea conserva sus puntos originales, sin importar cambios externos
